# Meta Data for Good - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 2, 11:00.** A participant asked whether we could work with Meta's
**AI for Good / Data for Good** datasets. Short answer: **yes, several of them, free and
with no application**, because Meta publishes them through the **Humanitarian Data
Exchange (HDX)**, which has an open API.

| Requested dataset | In class? | How |
|---|---|---|
| **Relative Wealth Index** | **yes** | free per-country CSVs on HDX |
| **Movement Distribution Maps** | **yes** | free on HDX, CC BY, updated through June 2026 |
| Activity Space Maps | not directly | not published on HDX; via Meta's own portal |
| Points of Interest | not directly | not published on HDX under Meta |

And there is far more than the four: **224 datasets** are published under *AI for Good at
Meta*, including the Social Connectedness Index, Commuting Zones, High Resolution
Population Density Maps, and the COVID-19 Trends and Impact Survey.

> Captured example of what a coding agent (Codex, Claude Code, or Antigravity CLI)
> produces from the **Lane A** prompts.


## About this data source

**Meta Data for Good / AI for Good.** Meta turns its data into *aggregated,
privacy-protected* public goods and gives them away through the **Humanitarian Data
Exchange (HDX)**, the portal the UN and NGOs use, which has a free open API.

- **Relative Wealth Index**: how well-off each ~2.4 km square is. A useful *confounder*:
  wealth shapes housing, water storage and care-seeking.
- **Movement Distribution**: how far people travel from home, by region and day. The
  behaviour channel disease travels through.

- **Explore it in a browser:** <https://data.humdata.org/organization/meta> (all 224
  datasets, each with a Download button)
- Meta's own write-ups: <https://dataforgood.facebook.com/>

> Two of the four datasets asked about are here (RWI, Movement Distribution). Activity Space
> Maps and Points of Interest are not on HDX and go through Meta's own portal.


## Step 0: helpers


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os, json, io
import urllib.request, urllib.parse

UA = {'User-Agent': 'SISMID2026-course/1.0 (your-email@example.com)'}
HDX = 'https://data.humdata.org/api/3/action/'

def cache_path(fname):
    for p in (f'../data/{fname}', f'data/{fname}', f'./{fname}'):
        if os.path.exists(p): return p
    return None

def hdx(action, **params):
    """Query the Humanitarian Data Exchange (CKAN) API. Free, no key."""
    url = HDX + action + '?' + urllib.parse.urlencode(params)
    return json.loads(urllib.request.urlopen(
        urllib.request.Request(url, headers=UA), timeout=60).read())['result']


## Step 1: discover what Meta publishes (HDX API, no key)

Rather than trusting a webpage, query the catalogue directly. This is the same
"ask the source what it has" move we use for every stream.

**Prompt used (Lane A):**

> *Using the HDX CKAN API (https://data.humdata.org/api/3/action/, free, no key), search*
> *package_search with fq='organization:meta' and list what 'AI for Good at Meta'*
> *publishes: title, resource formats, number of resources. How many datasets are there?*


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
try:
    res = hdx('package_search', fq='organization:meta', rows=50)
    cat = pd.DataFrame([{'title': p['title'],
                         'formats': ','.join(sorted({r.get('format','?')
                                                     for r in p.get('resources',[])})),
                         'n_resources': len(p.get('resources', []))}
                        for p in res['results']])
    print(f"{res['count']} datasets published by 'AI for Good at Meta' on HDX")
except Exception as e:
    print('HDX catalogue query failed:', e)
    cat = pd.DataFrame()
cat.head(12)


## Step 2: Relative Wealth Index (Mexico)

RWI estimates **relative wealth at ~2.4 km resolution** for low- and middle-income
countries, built from satellite imagery and connectivity data. We take Mexico, which
pairs with the dengue exercise: is dengue burden related to wealth?

**Prompt used (Lane A):**

> *From the HDX dataset 'relative-wealth-index', download the Mexico CSV*
> *(Mexico_relative_wealth_index.csv), report the number of grid cells and the rwi range,*
> *and make a scatter map coloured by rwi.*


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
COUNTRY = 'Mexico'
try:
    pkg = hdx('package_show', id='relative-wealth-index')
    url = next(r['url'] for r in pkg['resources']
               if r['name'] == f'{COUNTRY}_relative_wealth_index.csv')
    raw = urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=180).read()
    rwi = pd.read_csv(io.BytesIO(raw))
except Exception as e:
    p = cache_path('meta_rwi_mexico.csv')
    print('Live HDX pull failed:', e, '-> cache', p)
    rwi = pd.read_csv(p)

print(f'{len(rwi):,} grid cells; columns: {list(rwi.columns)}')
print(f"rwi range {rwi['rwi'].min():.2f} to {rwi['rwi'].max():.2f}")
rwi.head()


In [ ]:
plt.figure(figsize=(7.5,6))
s = plt.scatter(rwi['longitude'], rwi['latitude'], c=rwi['rwi'],
                s=1.5, cmap='viridis')
plt.colorbar(s, label='relative wealth index')
plt.xlabel('longitude'); plt.ylabel('latitude')
plt.title(f'Meta Relative Wealth Index, {COUNTRY} (~2.4 km grid)')
plt.tight_layout(); plt.show()


## Step 3: Movement Distribution (how far from home)

Movement Distribution gives, per admin region and day, the **fraction of pings falling
into distance-from-home bands**: `0` (stayed home), `(0, 10)` km, `[10, 100)` km, `100+`.
It is a behaviour signal, and it is **current** (updated through June 2026).

The live file is ~195 MB for all countries, so `LIVE = False` by default and we use the
Mexico slices cached in `data/`.

**Prompt used (Lane A):**

> *From the HDX dataset 'movement-distribution', get the distance-from-home distribution*
> *for Mexico. Note the global file is ~195 MB, so use the cached Mexico slices in data/*
> *unless I ask for a live rebuild. Plot the mean fraction per distance band over time,*
> *and rank the most home-bound and most mobile municipalities on the latest date.*


In [ ]:
# --- Produced by the agent from the Plan A / Step 3 prompt ---
LIVE = False   # True downloads the ~195 MB global file and filters to MEX
if LIVE:
    pkg = hdx('package_show', id='movement-distribution')
    url = next(r['url'] for r in pkg['resources'] if r['format'] == 'CSV')
    raw = urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=600).read()
    allc = pd.read_csv(io.BytesIO(raw))
    mex = allc[allc['country'] == 'MEX']
    daily = (mex.groupby(['ds','home_to_ping_distance_category'])
                ['distance_category_ping_fraction'].mean().reset_index()
                .rename(columns={'home_to_ping_distance_category':'distance_category',
                                 'distance_category_ping_fraction':'mean_fraction'}))
else:
    daily = pd.read_csv(cache_path('meta_movement_mx_national_daily.csv'))
    print('Using cached Mexico slices')

daily['ds'] = pd.to_datetime(daily['ds'])
print(f"{daily['ds'].nunique()} days: {daily['ds'].min().date()} -> {daily['ds'].max().date()}")
daily.head()


In [ ]:
piv = daily.pivot(index='ds', columns='distance_category', values='mean_fraction')
plt.figure(figsize=(10,4))
for c in piv.columns:
    plt.plot(piv.index, piv[c], marker='o', ms=3, label=f'{c} km')
plt.ylabel('mean fraction of pings'); plt.xlabel('date')
plt.title('Mexico: distance from home (Meta Movement Distribution)')
plt.legend(title='distance band'); plt.tight_layout(); plt.show()
print('mean share who stayed home:', round(piv['0'].mean(), 3))


In [ ]:
muni = pd.read_csv(cache_path('meta_movement_mx_municipal_latest.csv'))
home = (muni[muni['home_to_ping_distance_category'] == '0']
        .sort_values('distance_category_ping_fraction', ascending=False))
print(f"{muni['gadm_name'].nunique():,} municipalities on {muni['ds'].iloc[0]}")
print('\nmost home-bound municipalities:')
print(home[['gadm_name','distance_category_ping_fraction']].head(8).to_string(index=False))
print('\nmost mobile municipalities:')
print(home[['gadm_name','distance_category_ping_fraction']].tail(8).to_string(index=False))


## Step 4: sanity-check, save, and the honest limits

**Prompt used (Lane A):**

> *Save both to tidy CSVs, report missing values, and summarise the RWI error column.*


In [ ]:
rwi.to_csv('meta_rwi.csv', index=False)
daily.to_csv('meta_movement_daily.csv', index=False)
print('saved meta_rwi.csv and meta_movement_daily.csv')
print('RWI missing values:'); print(rwi.isna().sum())
print('\nRWI error column: mean', round(rwi['error'].mean(), 3),
      '-> every cell is a MODEL ESTIMATE with uncertainty, not a measurement')


## Reflection: what these are, and are not

- **RWI is a model output, not ground truth.** It is estimated from imagery and
  connectivity, and every cell ships an `error` column. Treat it as a covariate, and
  never as a measured income.
- **Movement Distribution is Facebook users with location history on**, aggregated and
  privacy-protected. That is a biased sample of the population, and the bias differs by
  country and by age.
- **Both are context, not case counts.** RWI is a plausible *confounder* for the dengue
  exercise (wealth relates to housing, water storage, and care-seeking), and movement is
  the behaviour channel through which disease spreads.
- **Two of the four requested datasets are not on HDX** (Activity Space Maps, Points of
  Interest). Those go through Meta's own portal and may need an agreement, so they are a
  good capstone stretch rather than an in-class pull.

**Stretch:** join RWI to Mexican states and compare against the dengue case data from
the Day 1 exercise. Does dengue burden track wealth?
